In [1]:
# @title Step 0: Setup and Installation
# Install ADK and LiteLLM for multi-model support

!pip install google-adk -q
!pip install litellm -q

print("Installation complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 15.4 MB/s eta 0:00:00
Installation complete.


In [2]:
# @title Import necessary libraries
import os
import asyncio
from google.adk.agents import Agent, SequentialAgent
from google.adk.models.lite_llm import LiteLlm # For multi-model support
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types # For creating message Content/Parts
from google.adk.tools import AgentTool
from google.adk.tools import google_search

import warnings
# Ignore all warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.ERROR)

print("Libraries imported.")

Libraries imported.


In [3]:
# @title Configure API Keys (Replace with your actual keys!)

# --- IMPORTANT: Replace placeholders with your real API keys ---
from google.colab import userdata

# Gemini API Key (Get from Google AI Studio: https://aistudio.google.com/app/apikey)
os.environ["GOOGLE_API_KEY"] = userdata.get('gemini_key')

# [Optional]
# OpenAI API Key (Get from OpenAI Platform: https://platform.openai.com/api-keys)
os.environ['OPENAI_API_KEY'] = userdata.get('openai_key')

# [Optional]
# Anthropic API Key (Get from Anthropic Console: https://console.anthropic.com/settings/keys)
os.environ['ANTHROPIC_API_KEY'] = 'YOUR_ANTHROPIC_API_KEY' # <--- REPLACE

os.environ['API_KEY_IN_USE'] = os.environ["GOOGLE_API_KEY"]

# --- Verify Keys (Optional Check) ---
print("API Keys Set:")
print(f"Google API Key set: {'Yes' if os.environ.get('GOOGLE_API_KEY') and os.environ['GOOGLE_API_KEY'] != 'YOUR_GOOGLE_API_KEY' else 'No (REPLACE PLACEHOLDER!)'}")
print(f"OpenAI API Key set: {'Yes' if os.environ.get('OPENAI_API_KEY') and os.environ['OPENAI_API_KEY'] != 'YOUR_OPENAI_API_KEY' else 'No (REPLACE PLACEHOLDER!)'}")
print(f"Anthropic API Key set: {'Yes' if os.environ.get('ANTHROPIC_API_KEY') and os.environ['ANTHROPIC_API_KEY'] != 'YOUR_ANTHROPIC_API_KEY' else 'No (REPLACE PLACEHOLDER!)'}")

# Configure ADK to use API keys directly (not Vertex AI for this multi-model setup)
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"


# @markdown **Security Note:** It's best practice to manage API keys securely (e.g., using Colab Secrets or environment variables) rather than hardcoding them directly in the notebook. Replace the placeholder strings above.

API Keys Set:
Google API Key set: Yes
OpenAI API Key set: Yes
Anthropic API Key set: No (REPLACE PLACEHOLDER!)


In [4]:
# --- Define Model Constants for easier use ---

# More supported models can be referenced here: https://ai.google.dev/gemini-api/docs/models#model-variations
MODEL_GEMINI_2_5_FLASH = "gemini-2.5-flash"

# More supported models can be referenced here: https://docs.litellm.ai/docs/providers/openai#openai-chat-completion-models
MODEL_GPT_4O = "openai/gpt-4.1" # You can also try: gpt-4.1-mini, gpt-4o etc.

# More supported models can be referenced here: https://docs.litellm.ai/docs/providers/anthropic
MODEL_CLAUDE_SONNET = "anthropic/claude-sonnet-4-20250514" # You can also try: claude-opus-4-20250514 , claude-3-7-sonnet-20250219 etc

print("\nEnvironment configured.")


Environment configured.


In [36]:
# @title Define the callbacks

from google.adk.models import LlmRequest, LlmResponse
from google.adk.agents.callback_context import CallbackContext
from google.genai import types

def log_agent_start(callback_context: CallbackContext) -> None:
    print(f"\n---  Starting Agent: {callback_context.agent_name} ---")

def log_agent_output(callback_context: CallbackContext) -> None:
    # Safely extract the text from the response parts
    agent_name = callback_context.agent_name

    # 2. Map the agent name to its specific output_key
    # (Or just look for the key you defined in your LlmAgent)
    output_keys = {
        "google_search_agent": "initial_draft",
        "critique_agent": "critique_notes"
    }

    key = output_keys.get(agent_name)

    if key:
        # Retrieve the data from the shared session state
        output = callback_context.state.get(key, "No output found in state.")
        print(f"---  {agent_name} Finished ---")
        print(f"Data saved to '{key}': {str(output)[:200]}...")
        print("-" * 30)

    return None

In [37]:
# @title Define the Agents
# Use one of the model constants defined earlier
AGENT_MODEL = MODEL_GEMINI_2_5_FLASH # Starting with Gemini

google_search_agent = Agent(
    name="google_search_agent",
    model=AGENT_MODEL,
    output_key="initial_draft",
    instruction="Search for the most up-to-date answer to the user's question. Provide a detailed draft.",
    tools=[google_search],
    before_agent_callback=log_agent_start,
    after_agent_callback=log_agent_output,
)

critique_agent = Agent(
    name="critique_agent",
    model=AGENT_MODEL,
    output_key="critique_notes",
    instruction="""
    Review the following draft: {initial_draft}
    Check for accuracy, tone, and completeness.
    List specific suggestions for improvement or identify missing facts.
    """,
    before_agent_callback=log_agent_start,
    after_agent_callback=log_agent_output,
)

refine_agent = Agent(
    name="refine_agent",
    model=AGENT_MODEL,
    instruction="""
    Using the initial draft: {initial_draft}
    And the following critique: {critique_notes}
    Rewrite the answer to be professional, accurate, and concise.
    This will be the final version shown to the user.
    """,
    before_agent_callback=log_agent_start,
    after_agent_callback=log_agent_output,
)

main_workflow = SequentialAgent(
    name="search_pipeline",
    description="A pipeline that searches, critiques, and refines answers.",
    sub_agents=[
        google_search_agent,
        critique_agent,
        refine_agent
        ]
)

In [38]:
# @title Setup Session Service and Runner

# --- Session Management ---
# Key Concept: SessionService stores conversation history & state.
# InMemorySessionService is simple, non-persistent storage for this tutorial.
session_service = InMemorySessionService()

# Define constants for identifying the interaction context
APP_NAME = "search_app"
USER_ID = "user_1"
SESSION_ID = "session_005" # Using a fixed ID for simplicity

# Create the specific session where the conversation will happen
session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)
print(f"Session created: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

# --- Runner ---
# Key Concept: Runner orchestrates the agent execution loop.
runner = Runner(
    agent=main_workflow, # The agent we want to run
    app_name=APP_NAME,   # Associates runs with our app
    session_service=session_service # Uses our session manager
)
print(f"Runner created for agent '{runner.agent.name}'.")

Session created: App='search_app', User='user_1', Session='session_005'
Runner created for agent 'search_pipeline'.


In [39]:
# @title Define Agent Interaction Function

from google.genai import types # For creating message Content/Parts

async def call_agent_async(query: str, runner, user_id, session_id):
  """Sends a query to the agent and prints the final response."""
  print(f"\n>>> User Query: {query}")

  # Prepare the user's message in ADK format
  content = types.Content(role='user', parts=[types.Part(text=query)])

  final_response_text = "Agent did not produce a final response." # Default

  # Key Concept: run_async executes the agent logic and yields Events.
  # We iterate through events to find the final answer.
  async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
      # You can uncomment the line below to see *all* events during execution
      # print(f"  [Event] Author: {event.author}, Type: {type(event).__name__}, Final: {event.is_final_response()}, Content: {event.content}")

      # Key Concept: is_final_response() marks the concluding message for the turn.
      if event.is_final_response():
          if event.content and event.content.parts:
             # Assuming text response in the first part
             final_response_text = event.content.parts[0].text
          elif event.actions and event.actions.escalate: # Handle potential errors/escalations
             final_response_text = f"Agent escalated: {event.error_message or 'No specific message.'}"
          # Add more checks here if needed (e.g., specific error codes)
           # Stop processing events once the final response is found

  print(f"<<< Agent Response: {final_response_text}")

In [40]:
# @title Run the Initial Conversation

# We need an async function to await our interaction helper
async def run_conversation():
  await call_agent_async("What is a wormhole?",
                                       runner=runner,
                                       user_id=USER_ID,
                                       session_id=SESSION_ID)

# Execute the conversation using await in an async context (like Colab/Jupyter)
await run_conversation()


>>> User Query: What is a wormhole?

---  Starting Agent: google_search_agent ---
---  google_search_agent Finished ---
Data saved to 'initial_draft': A wormhole is a hypothetical "tunnel" or "bridge" through space-time that could create shortcuts for long journeys across the universe, theoretically connecting two disparate points in space, time, or...
------------------------------

---  Starting Agent: critique_agent ---
---  critique_agent Finished ---
Data saved to 'critique_notes': Here's a review of the draft, checking for accuracy, tone, and completeness, with specific suggestions for improvement:

**Overall Assessment:**
The draft is well-written, informative, and maintains a...
------------------------------

---  Starting Agent: refine_agent ---
<<< Agent Response: A wormhole is a hypothetical "tunnel" or "bridge" through space-time that could create shortcuts for long journeys across the universe, theoretically connecting two disparate points in space, time, or even differe